# Функция в переменной + `callable`

Разбираем строки из нашей игры (`ghost.py`):

```python
notify = getattr(self, "_appearance_notify", None)
if callable(notify):
    notify(self)
```

**Важно заранее:** отдельной функции с именем `notify` в проекте нет.  
`notify` — это **переменная**. В неё кладут ссылку на другую функцию (или пусто).

## Теория (коротко)

1. Функцию можно положить в переменную, как число:
   ```python
   def привет():
       print("hi")
   x = привет   # положили
   x()          # вызвали то, что лежит в x
   ```
2. Иногда в переменной лежит не функция, а `None` («пусто», звонить некому).
3. `callable(x)` — вопрос: «это можно вызвать через `x(...)`?»  
   - функция → `True`  
   - `None` → `False`
4. Поэтому сначала проверяем, потом вызываем — иначе будет ошибка.

In [ ]:
def сказать_привет():
    print("привет")

коробка = сказать_привет   # функция лежит в переменной
print("callable?", callable(коробка))
коробка()                  # вызов того, что в переменной

пусто = None
print("callable(None)?", callable(пусто))
# пусто()  ← так нельзя, будет TypeError

## Откуда берётся функция в игре

Настоящая функция — метод игры:

```python
# main_work.py
self.ghost_manager.appearance_callback = self.on_ghost_appeared_on_map
```

Менеджер на кадр кладёт её призраку:

```python
ghost._appearance_notify = self.appearance_callback
```

Если ещё не подключили — там `None`.

## Построчно: три строки призрака

```python
notify = getattr(self, "_appearance_notify", None)
if callable(notify):
    notify(self)
```

### Строка 1. `notify = getattr(self, "_appearance_notify", None)`

- Смотрим у призрака поле `_appearance_notify`.
- Нашли → кладём это значение в переменную `notify` (функция игры или `None`).
- Не нашли → в `notify` кладём запасное `None` (без ошибки).

Пока **ничего не вызывали** — только достали в переменную.

### Строка 2. `if callable(notify):`

- Вопрос: «то, что в `notify`, можно вызвать?»
- `True` — да, это функция.
- `False` — нет (часто `None`) → внутрь `if` не заходим.

### Строка 3. `notify(self)`

- Вызываем **то, что лежит в переменной**.
- По сути это звонок в `on_ghost_appeared_on_map(призрак)`.
- `self` — сам призрак: «вот я появился».

**Одной фразой:** достали → спросили «можно звать?» → если да, позвали.

In [ ]:
# Мини-демо той же логики

def on_ghost_appeared_on_map(призрак):
    print("Игра: появился", призрак, "→ −10 рассудка")


class Призрак:
    def стать_видимым(self):
        notify = getattr(self, "_appearance_notify", None)  # 1) достали
        if callable(notify):                                 # 2) можно вызвать?
            notify(self)                                     # 3) вызвали


print("Без подключения:")
g1 = Призрак()
g1.стать_видимым()  # тишина, ошибки нет

print("С подключением:")
g2 = Призрак()
g2._appearance_notify = on_ghost_appeared_on_map  # положили функцию в атрибут
g2.стать_видимым()

## Когда такое решение оправдано

Паттерн «положить функцию в переменную + вызвать, если `callable`» нужен не всегда.

**Оправдан, когда:**
- объект **не должен знать** про всю игру / UI (призрак не импортирует `Game`);
- реакция **необязательна**: нет обработчика → просто молчим, без ошибки;
- одно и то же событие могут обрабатывать **разные** функции (подключили другую — поведение сменилось);
- удобно в тестах / редакторе: модуль живёт один, игру рядом не обязали.

**Не нужен, когда:**
- реакция всегда одна и та же и оба класса и так связаны;
- хватает простого флага / прямого вызова `game.drain_sanity(...)` в том же месте;
- «положили функцию» ради красоты, а не ради слабой связи.

**В нашем ghost-event:** призрак ловит `INVISIBLE → IDLE` и только сообщает «я появился». Минус рассудка остаётся в игре — поэтому callback здесь уместен.

---

## Задание

Сделайте кнопку, которая умеет «мягко» звать обработчик.

1. Напишите функцию `открыть_журнал()`, которая печатает `журнал открыт`.
2. Создайте класс `Кнопка` с методом `нажали()`:
   - через `getattr` достаньте атрибут `on_click` (если нет — `None`);
   - если `callable` — вызовите его **без аргументов**.
3. Проверьте:
   - кнопка без `on_click` — ошибка не должна появиться;
   - потом положите функцию в переменную/атрибут: `кнопка.on_click = открыть_журнал` и снова нажмите — должна быть печать.

Допишите ячейку ниже.

In [ ]:
# TODO: ваше решение

def открыть_журнал():
    ...


class Кнопка:
    def нажали(self):
        ...


кнопка = Кнопка()
кнопка.нажали()  # без обработчика — тихо

кнопка.on_click = открыть_журнал  # положили функцию в переменную-атрибут
кнопка.нажали()  # ожидаем: журнал открыт

## Ответ на задание

In [ ]:
def открыть_журнал():
    print("журнал открыт")


class Кнопка:
    def нажали(self):
        обработчик = getattr(self, "on_click", None)
        if callable(обработчик):
            обработчик()


кнопка = Кнопка()
кнопка.нажали()  # тихо

кнопка.on_click = открыть_журнал  # функция лежит в переменной on_click
кнопка.нажали()  # журнал открыт

## Итог

| Кусок | Зачем |
|---|---|
| `notify = ...` | положили функцию (или пусто) в переменную |
| `callable(notify)` | спросили: можно ли вызывать? |
| `notify(self)` | вызвали только если да |

Пустота (`None`) нужна, чтобы «звонка нет» было безопасным состоянием, а не ошибкой.